[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/00_setup_verification.ipynb)

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar si estamos en Google Colab ──────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Estrategia 1: acceso por ID de carpeta raíz del proyecto
    _p1 = "/content/drive/.shortcut-targets-by-id/13VEeA8Jt0G_NNpJelWZrhacBhA_mRpnU"
    # Estrategia 2: acceso por ID de TrainData (subir un nivel)
    _p2 = "/content/drive/.shortcut-targets-by-id/13Dj5DYGOwgMGomMC8gCySKGr0ne8hOpU"

    if os.path.exists(_p1) and os.path.isdir(_p1):
        DATA_ROOT_OVERRIDE = _p1
        print(f"Estrategia 1 OK: {_p1}")
    elif os.path.exists(_p2) and os.path.isdir(_p2):
        DATA_ROOT_OVERRIDE = str(Path(_p2).parent)
        print(f"Estrategia 2 OK: {DATA_ROOT_OVERRIDE}")
    else:
        import subprocess
        _r = subprocess.run(
            ["find", "/content/drive/MyDrive", "-name", "TrainData",
             "-type", "d", "-maxdepth", "6"],
            capture_output=True, text=True, timeout=30
        )
        _hits = [p.replace('/TrainData', '') for p in _r.stdout.strip().splitlines() if p]
        if _hits:
            DATA_ROOT_OVERRIDE = _hits[0]
            print(f"Estrategia 3 OK: {DATA_ROOT_OVERRIDE}")
        else:
            print("No se encontró el dataset. Ajusta DATA_ROOT_OVERRIDE manualmente:")
            print("  DATA_ROOT_OVERRIDE = 'ruta/en/tu/Drive'")

    if DATA_ROOT_OVERRIDE:
        print(f"Dataset: {DATA_ROOT_OVERRIDE}")
else:
    print('Entorno local — se usará detección automática de rutas.')


# 00 — Verificación del Entorno

**Proyecto:** Detección de Deslizamientos con Deep Learning — Landslide4Sense  
**Objetivo:** Confirmar que todas las dependencias están correctamente instaladas y que el dataset es accesible.

---

## 1. Instalar dependencias

In [ ]:
# Ejecutar solo si es necesario
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install segmentation-models-pytorch timm h5py tqdm tensorboard torchmetrics pyyaml

## 2. Verificar versiones de librerías

In [ ]:
import sys
import importlib

libs = {
    "torch":                    "PyTorch",
    "torchvision":              "TorchVision",
    "segmentation_models_pytorch": "SMP (U-Net)",
    "timm":                     "timm (EfficientNet)",
    "h5py":                     "h5py",
    "numpy":                    "NumPy",
    "sklearn":                  "Scikit-learn",
    "matplotlib":               "Matplotlib",
    "tqdm":                     "tqdm",
}

print(f"Python: {sys.version.split()[0]}\n{'─'*40}")
for lib, name in libs.items():
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, "__version__", "?")
        print(f"  ✓  {name:<30} {ver}")
    except ImportError:
        print(f"  ✗  {name:<30} NO INSTALADO")

## 3. Verificar disponibilidad de GPU

In [ ]:
import torch

print(f"PyTorch:         {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# MPS (Apple Silicon)
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("MPS (Apple Silicon): disponible")

device = "cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends,"mps") and torch.backends.mps.is_available() else "cpu"
print(f"\nDispositivo seleccionado: {device}")

## 4. Verificar estructura del dataset

In [ ]:
import os, sys
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL DATASET
# Si la celda de Drive de arriba encontró el dataset, DATA_ROOT_OVERRIDE
# ya está definida. Si no, puedes ajustarla manualmente aquí:
# DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/landslide4sense"
# ══════════════════════════════════════════════════════════════════════════════

# Preservar DATA_ROOT_OVERRIDE si ya fue fijada por la celda de Drive
try:
    DATA_ROOT_OVERRIDE
except NameError:
    DATA_ROOT_OVERRIDE = None

# ── Detección automática ──────────────────────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    print(f"Usando ruta configurada: {DATA_ROOT}")
else:
    _cwd = Path(os.getcwd())
    _candidates = [
        _cwd / ".." / "data",
        _cwd / "..",
        _cwd / "data",
        _cwd,
        Path("/content/drive/MyDrive/landslide4sense"),
        Path("/content/drive/MyDrive/Landslide_ML"),
        Path("/content/landslide4sense"),
        Path("/content/data"),
        Path("/content"),
    ]
    DATA_ROOT = None
    for _c in _candidates:
        if (_c / "TrainData" / "img").exists():
            DATA_ROOT = _c.resolve()
            print(f"Dataset detectado automáticamente en: {DATA_ROOT}")
            break

if DATA_ROOT is None or not (DATA_ROOT / "TrainData" / "img").exists():
    print("""
⚠️  Dataset no encontrado. Opciones:

━━━  OPCIÓN A — Google Drive (recomendado)  ━━━
  Ejecuta la celda de Drive de arriba y vuelve a correr esta celda.

━━━  OPCIÓN B — Kaggle API  ━━━
  !pip install kaggle -q
  # Sube tu kaggle.json y ejecuta:
  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d landslide4sense/competition -p /content/
  !unzip -q /content/competition.zip -d /content/landslide4sense/
  DATA_ROOT_OVERRIDE = "/content/landslide4sense"
""")
    raise FileNotFoundError("Dataset no encontrado. Establece DATA_ROOT_OVERRIDE con la ruta correcta.")

# ── Verificar estructura del dataset ─────────────────────────────────────────
n_train = len(list((DATA_ROOT / "TrainData" / "img").glob("*.h5")))
print(f"✓ Dataset encontrado en: {DATA_ROOT}")
print(f"  TrainData/img : {n_train} archivos .h5")

partitions = {
    "TrainData": {"img": True,  "mask": True},
    "ValidData": {"img": True,  "mask": False},
    "TestData":  {"img": True,  "mask": False},
}

all_ok = True
for part, dirs in partitions.items():
    for subdir, required in dirs.items():
        p = DATA_ROOT / part / subdir
        n = len(list(p.glob("*.h5"))) if p.exists() else -1
        status = "✓" if n > 0 else ("✗" if required else "—")
        print(f"  {status}  {part}/{subdir:<10} → {n if n >= 0 else 'no existe':>5} archivos .h5")
        if required and n <= 0:
            all_ok = False

if not all_ok:
    print("\n⚠️  Dataset incompleto. Ver data/README.md para instrucciones de descarga.")
else:
    print("\n✓ Dataset correctamente estructurado.")


## 5. Verificar forma y tipo de un parche

In [ ]:
import h5py
import numpy as np

img_dir  = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"

img_files = sorted(img_dir.glob("*.h5"))
if not img_files:
    raise FileNotFoundError(f"No se encontraron archivos .h5 en {img_dir}\n"
                            f"Verifica que DATA_ROOT sea correcto: {DATA_ROOT}")
sample_img  = img_files[0]
sample_mask = mask_dir / sample_img.name

with h5py.File(sample_img, "r") as f:
    key   = list(f.keys())[0]
    patch = f[key][()]

with h5py.File(sample_mask, "r") as f:
    mkey = list(f.keys())[0]
    mask = f[mkey][()]

print(f"Archivo imagen:  {sample_img.name}")
print(f"Forma del parche: {patch.shape}  dtype={patch.dtype}")
print(f"Rango de valores: [{patch.min():.4f}, {patch.max():.4f}]")
print(f"\nArchivo máscara: {sample_mask.name}")
print(f"Forma de máscara: {mask.shape}  dtype={mask.dtype}")
print(f"Valores únicos:   {np.unique(mask)}")
print(f"¿Deslizamiento?   {'Sí' if mask.max() > 0 else 'No'}")
print(f"\n✓ Estructura de datos verificada — Listo para entrenar.")

## 6. Test rápido del módulo `src/`

In [ ]:
import sys
sys.path.insert(0, "..")

from src.config import TrainingConfig, get_debug_config
from src.dataset import Landslide4SenseDataset
from src.models import build_model, model_summary
from src.utils import set_seed, get_device

cfg = get_debug_config()
cfg.data_root = str(DATA_ROOT)

set_seed(cfg.seed)
device = get_device()
print(f"Dispositivo: {device}")

# Construir modelo de prueba
model = build_model("resnet50", n_channels=14, pretrained=False)
print(model_summary(model, n_channels=14))

# Cargar un batch de prueba
ds = Landslide4SenseDataset(
    img_dir=str(DATA_ROOT / "TrainData" / "img"),
    mask_dir=str(DATA_ROOT / "TrainData" / "mask"),
    indices=list(range(8)),
    normalize=True,
)
sample = ds[0]
print(f"\nBatch de prueba:")
print(f"  image shape: {sample['image'].shape}")
print(f"  label:       {sample['label'].item()}")
print(f"  filename:    {sample['filename']}")
print("\n✓ Módulo src/ funcionando correctamente.")

---

## Resumen

Si todas las celdas se ejecutaron sin errores, el entorno está listo para:

1. `01_eda_analysis.ipynb` — Análisis exploratorio de datos
2. `02_preprocessing.ipynb` — Preprocesamiento y augmentación
3. `03_baseline_rf.ipynb` — Baseline Random Forest
4. `04_resnet50.ipynb` — Fine-tuning ResNet-50
5. `05_efficientnet_b4.ipynb` — Fine-tuning EfficientNet-B4
6. `06_unet_segmentation.ipynb` — Segmentación con U-Net
7. `07_evaluation_comparison.ipynb` — Comparativa final